# Analyses: results by `recognized`, hierarchy gain by group

From the saved checkpoint predictions only. **Nothing is trained, so no GPU is needed.** Validation data only.

1. Every headline comparison on recognised and unrecognised drawings separately (thresholds from the whole
   calibration half, as in the main results).
2. Where the semantic tree's gain over flat answers comes from, per group of the frozen tree.

**Before running:**
1. In **Input**, the `AI_pictionary` notebook output must be the **followup_v2 version** (it holds
   `runs/predictions_all/*.npz`, 19 files). Use **⋮ → Check for updates** if needed.
2. Keep your Kaggle Dataset attached and **Internet on**.
3. Choose **Accelerator: None**, then **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed")

candidates = [p.parent for p in Path("/kaggle/input").rglob("gru_anytime_uniform_s42.npz")]
if not candidates:
    raise RuntimeError("followup_v2 predictions not found: attach the followup_v2 version of the AI_pictionary output")
PREDICTIONS = max(candidates, key=lambda d: len(list(d.glob("*.npz"))))
print("predictions:", PREDICTIONS, len(list(PREDICTIONS.glob("*.npz"))), "files")


In [ ]:
!PYTHONPATH=src python -m quickdraw_exp.analysis --predictions {PREDICTIONS} --out /kaggle/working/runs/analysis_v1


In [ ]:
import json
import numpy as np

r = json.loads(Path("/kaggle/working/runs/analysis_v1/results.json").read_text())
print(f"recognized share of validation drawings: {r['recognized_share_val']:.3f}")
print("\nAUC by subgroup (a vs b, difference):")
for row in r["by_recognized"]["comparisons"]:
    if row.get("missing"):
        print(f"  {row['name']}: not available")
        continue
    cells = "  ".join(f"{g}: {row[g]['a']:.4f} vs {row[g]['b']:.4f} ({row[g]['difference']:+.4f})" for g in ("all", "recognized", "unrecognized"))
    print(f"  {row['name']}\n      {cells}")
print("\nPer run: drawings, end-of-drawing accuracy and AUC (pmax) by subgroup:")
for run, groups in r["by_recognized"]["per_run"].items():
    print(f"  {run:30s}", "  ".join(f"{g} n={v['drawings']} end {v['end_accuracy']:.4f} auc {v['pmax']['auc']:.4f}" for g, v in groups.items()))

print("\nHierarchy gain by group (precision 0.95, semantic minus flat reward area):")
for run, h in r["hierarchy_by_group"].items():
    print(f"\n  {run}: overall gain {h['gain']:+.4f}")
    for group, g in sorted(h["by_group"].items(), key=lambda kv: -kv[1]["gain"]):
        print(f"    {group:32s} classes {g['classes']:2d}  gain {g['gain']:+.4f}  share {g['share_of_total_gain']:.2f}  "
              f"answers at subgroup {g['answers_at_subgroup']:.3f}  at group {g['answers_at_group']:.3f}")
    print("    top classes:", [(c, round(v, 3)) for c, v in h["top_classes"]])
    print("    bottom classes:", [(c, round(v, 3)) for c, v in h["bottom_classes"]])
